# Bronze Layer Validation

This notebook validates the Bronze layer implementation for all domains:

- market_prices  
- weather  
- grid_events  
- reference  

Validation includes:
- table existence
- row counts
- schema verification
- ingestion metadata presence
- domain integrity

In [0]:
import yaml

config_path = "/Workspace/Repos/Mini Projects/Vattenfall-Engineering-Capstone-Project/config/project_config.yml"

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

catalog = config["catalog"]
raw_schema = config["schemas"]["raw"]

domains = config["domains"]

print("Catalog:", catalog)
print("Schema:", raw_schema)
print("Domains:", domains)

## 1. Validate Table Existence

In [0]:
spark.sql(f"SHOW TABLES IN {catalog}.{raw_schema}").show(truncate=False)

In [0]:
for d in domains:
    table = f"{catalog}.{raw_schema}.bronze_{d}"
    try:
        count = spark.table(table).count()
        print(f"{table} → {count} rows")
    except:
        print(f"{table} → NOT FOUND")

## 3. Validate Schema (Grid Events Example)

In [0]:
spark.table(f"{catalog}.{raw_schema}.bronze_grid_events").printSchema()

## 4. Validate Data Preview

In [0]:
display(spark.table(f"{catalog}.{raw_schema}.bronze_grid_events"))

## 5. Validate Ingestion Metadata

In [0]:
display(
    spark.table(f"{catalog}.{raw_schema}.bronze_grid_events")
    .select("ingestion_ts", "source_system")
)

In [0]:
display(
    spark.table(f"{catalog}.{raw_schema}.bronze_grid_events")
    .select("source_system")
    .distinct()
)

In [0]:
display(
    spark.table(f"{catalog}.{raw_schema}.bronze_grid_events")
    .filter("event_id IS NULL")
)

## 8. Validate All Domains Summary

In [0]:
for d in domains:
    table = f"{catalog}.{raw_schema}.bronze_{d}"
    try:
        df = spark.table(table)
        print(f"\n🔹 {table}")
        print("Rows:", df.count())
        print("Columns:", len(df.columns))
        df.show(3)
    except Exception as e:
        print(f"{table} → ERROR:", e)